# Uji Model - Anggota 1 (Ketua)

**Mazmur Bryan**: EncoderCNN, LSTMCell manual, DecoderLSTM, CaptioningModel.

Semua modul dibangun from scratch (tanpa model pretrained, tanpa `nn.LSTM`) dan diuji dengan tensor acak, karena dataset belum dibutuhkan untuk memastikan arsitektur benar.

Cara menjalankan: **Runtime → Run all** (runtime T4 GPU).

## Setup

In [ ]:
# Sambungkan Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Ambil kode dari GitHub
import os
REPO_URL = 'https://github.com/steezym/image-captioning.git'
PATH = '/content/image-captioning'

if not os.path.exists(PATH):
    !git clone -q {REPO_URL} {PATH}
%cd {PATH}

# Pindah ke branch kerja (ganti sesuai peran), lalu ambil versi terbaru
BRANCH = 'main'   # Anggota 1: 'anggota1-model', Anggota 2: 'anggota2-data', dst.
!git fetch -q
!git checkout -q {BRANCH}
!git pull -q origin {BRANCH}

# Muat ulang kode terbaru tanpa restart runtime.
# (%autoreload tidak dipakai karena gagal di Python 3.13: modul 'imp' sudah dihapus)
# Cara pakai setelah git pull:  muat_ulang()  lalu import ulang modulnya
import sys
def muat_ulang():
    for nama in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
        del sys.modules[nama]
    print('Modul src dimuat ulang pada import berikutnya')

DRIVE_ROOT = '/content/drive/MyDrive/ImageCaptioning'
print('Drive OK:', os.path.exists(DRIVE_ROOT))

## Cek GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. EncoderCNN

Vanilla CNN 5 blok (Conv 3×3 → BatchNorm → ReLU → MaxPool) + Global Average Pooling.
Gambar `(B, 3, 128, 128)` → fitur `(B, 256)`.

Yang dicek: shape tiap blok, keluaran `(4, 256)`, dan gradien mengalir sampai lapisan pertama.

In [ ]:
!python -m tests.test_encoder

## 2. LSTMCell manual

Sel LSTM ditulis sendiri dari `nn.Linear`: 4 gate (input, forget, candidate, output), bias forget gate = 1.

Yang dicek: shape, rumus gate dihitung ulang secara terpisah, bias forget, rentang `h`, dan gradien setelah 30 langkah.

In [ ]:
!python -m tests.test_lstm_cell

## 3. DecoderLSTM

Embedding → LSTMCell → Dropout → Linear. Fitur gambar menjadi memori awal (`h0`, `c0`). Training memakai teacher forcing, pembuatan caption memakai greedy search.

Yang dicek: shape `(4, 29, vocab)`, tidak mengintip kata masa depan, fitur gambar dipakai, embedding `<pad>` = 0, gradien, dan `generate`.

In [ ]:
!python -m tests.test_decoder

## 4. CaptioningModel + uji hafalan

Encoder + decoder digabung dalam satu model. **Uji hafalan**: model dilatih 150 langkah pada 4 pasangan gambar-caption acak. Jika semua bagian tersambung benar, loss turun drastis dan model bisa menulis ulang caption yang dihafal.

In [ ]:
!python -m tests.test_model

## Ringkasan

Jika keempat uji menampilkan `LULUS`, arsitektur model siap dipakai untuk training dengan data Flickr8k (Anggota 2 & 3).